

#1. Cargas



In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

caminho_pasta = '/content/drive/MyDrive/Projetos/RecomendacaoSpotify'

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# Carga
df_musicas = pd.read_parquet(f'{caminho_pasta}/df_musicas.parquet')
# Carregar metadata sem a coluna popularidade, já que está zerada
metadata_api = pd.read_csv(f'{caminho_pasta}/metadata_enriquecido.csv').drop(columns=['popularidade'], errors='ignore')

# Garantir tipo correto na chave de join
metadata_api['spotify_track_uri'] = metadata_api['spotify_track_uri'].astype(str)
df_musicas['spotify_track_uri'] = df_musicas['spotify_track_uri'].astype(str)

print(f'df_musicas:    {df_musicas.shape}')
print(f'metadata_api:  {metadata_api.shape}')

df_musicas:    (24040, 30)
metadata_api:  (6185, 4)


#2. Recuperar dados trazidos pelo Spotify

In [ ]:
df_features_plays = df_musicas.merge(metadata_api, on='spotify_track_uri', how='left')

uris_unicas        = df_features_plays['spotify_track_uri'].nunique()
uris_com_api       = df_features_plays[df_features_plays['duracao_ms'].notna()]['spotify_track_uri'].nunique()
uris_sem_api       = uris_unicas - uris_com_api
cobertura          = uris_com_api / uris_unicas * 100

print(f'URIs únicas no histórico:  {uris_unicas}')
print(f'Com dados da API:          {uris_com_api} ({cobertura:.1f}%)')
print(f'Sem dados da API (proxy):  {uris_sem_api} ({100 - cobertura:.1f}%)')

URIs únicas no histórico:  5374
Com dados da API:          5374 (100.0%)
Sem dados da API (proxy):  0 (0.0%)


In [ ]:
import re

def parse_data_release(valor):
    if pd.isna(valor):
        return pd.NaT
    valor = str(valor).strip()
    if re.match(r'^\d{4}$', valor):
        return pd.to_datetime(valor + '-01-01')
    if re.match(r'^\d{4}-\d{2}$', valor):
        return pd.to_datetime(valor + '-01')
    if re.match(r'^\d{4}-\d{2}-\d{2}$', valor):
        return pd.to_datetime(valor)
    return pd.NaT

metadata_api['data_release'] = metadata_api['data_release'].apply(parse_data_release)

# Diagnóstico
nulos = metadata_api['data_release'].isna().sum()
print(f"Datas parseadas com sucesso: {len(metadata_api) - nulos}/{len(metadata_api)}")
print(f"Formato da coluna: {metadata_api['data_release'].dtype}")
print(metadata_api['data_release'].describe())

Datas parseadas com sucesso: 6185/6185
Formato da coluna: datetime64[ns]
count                             6185
mean     2000-02-17 09:22:15.812449408
min                1958-01-01 00:00:00
25%                1989-01-01 00:00:00
50%                1999-04-19 00:00:00
75%                2014-11-25 00:00:00
max                2026-04-24 00:00:00
Name: data_release, dtype: object


#3. Duração de uma música seguindo hierarquia:

1 - Dados vindos do Spotify

2 - Proxy por trackdone

3 - Proxy por medianas

In [ ]:
# Levantamento de trackdone por URI
trackdone_por_uri = (
    df_features_plays[df_features_plays['reason_end'] == 'trackdone']
    .groupby('spotify_track_uri')['ms_played']
    .agg(
        count='count',
        mediana='median',
        minimo='min',
        maximo='max',
        std='std'
    )
    .reset_index()
)

# Trazer nome da música e artista
nomes = df_features_plays[['spotify_track_uri', 'musica_unificada', 'artista_unificado']].drop_duplicates('spotify_track_uri')
trackdone_por_uri = trackdone_por_uri.merge(nomes, on='spotify_track_uri', how='left')

print(f"URIs com pelo menos 1 trackdone: {len(trackdone_por_uri)}")
print(f"URIs sem nenhum trackdone: {df_features_plays['spotify_track_uri'].nunique() - len(trackdone_por_uri)}")
print()
print("Distribuição de quantidade de trackdones por URI:")
print(trackdone_por_uri['count'].describe())
print()
print("URIs com apenas 1 trackdone:", (trackdone_por_uri['count'] == 1).sum())
print("URIs com 2 ou mais trackdones:", (trackdone_por_uri['count'] >= 2).sum())
print()
print("Casos suspeitos (mediana < 30s):")
display(trackdone_por_uri[trackdone_por_uri['mediana'] < 30000][
    ['musica_unificada', 'artista_unificado', 'count', 'mediana', 'minimo', 'maximo']
])

URIs com pelo menos 1 trackdone: 4222
URIs sem nenhum trackdone: 1152

Distribuição de quantidade de trackdones por URI:
count    4222.000000
mean        3.037423
std         5.015307
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        57.000000
Name: count, dtype: float64

URIs com apenas 1 trackdone: 2283
URIs com 2 ou mais trackdones: 1939

Casos suspeitos (mediana < 30s):


,musica_unificada,artista_unificado,count,mediana,minimo,maximo
423,Tempo Perdido,Legião Urbana,1,19602.0,19602,19602
675,É Tudo Pra Ontem,Emicida,3,28750.0,22235,310400
2560,Speed,ANGRA,1,6406.0,6406,6406
2772,Speak to Me,Pink Floyd,1,15738.0,15738,15738
2843,You've Got To Be Crazy,Pink Floyd,1,16501.0,16501,16501


In [ ]:
# Fonte 2: mediana trackdone com filtro de artefatos
duracao_trackdone = (
    df_features_plays[
        (df_features_plays['reason_end'] == 'trackdone') &
        (df_features_plays['ms_played'] >= 10000)
    ]
    .groupby('spotify_track_uri')['ms_played']
    .median()
    .rename('duracao_trackdone_ms')
    .reset_index()
)

# Montar duracao_estimada_ms com hierarquia de fontes
df_features_plays['duracao_estimada_ms'] = df_features_plays['duracao_ms']
df_features_plays['fonte_duracao'] = np.where(df_features_plays['duracao_ms'].notna(), 'api', None)

# Fonte 2: preencher onde API não tem
df_features_plays = df_features_plays.merge(duracao_trackdone, on='spotify_track_uri', how='left')
mask_fonte2 = df_features_plays['duracao_estimada_ms'].isna() & df_features_plays['duracao_trackdone_ms'].notna()
df_features_plays.loc[mask_fonte2, 'duracao_estimada_ms'] = df_features_plays.loc[mask_fonte2, 'duracao_trackdone_ms']
df_features_plays.loc[mask_fonte2, 'fonte_duracao'] = 'trackdone'

# Remover coluna auxiliar
df_features_plays = df_features_plays.drop(columns='duracao_trackdone_ms')

# Auditoria
print("Distribuição por fonte de duração (execuções):")
print(df_features_plays['fonte_duracao'].value_counts(dropna=False))
print()
print("Distribuição por fonte de duração (URIs únicas):")
print(df_features_plays.groupby('fonte_duracao', dropna=False)['spotify_track_uri'].nunique())

Distribuição por fonte de duração (execuções):
fonte_duracao
api    24040
Name: count, dtype: int64

Distribuição por fonte de duração (URIs únicas):
fonte_duracao
api    5374
Name: spotify_track_uri, dtype: int64


In [ ]:
# Fonte 3: Mediana global (apenas trackdone >= 10s)
mediana_global = df_features_plays[
    (df_features_plays['reason_end'] == 'trackdone') &
    (df_features_plays['ms_played'] >= 10000)
]['ms_played'].median()

print(f"Mediana global: {mediana_global/60000:.2f} min")

# Mediana por artista (apenas trackdone >= 10s)
duracao_por_artista_proxy = (
    df_features_plays[
        (df_features_plays['reason_end'] == 'trackdone') &
        (df_features_plays['ms_played'] >= 10000)
    ]
    .groupby('artista_unificado')['ms_played']
    .agg(mediana='median', std='std')
    .reset_index()
)
duracao_por_artista_proxy['cv'] = duracao_por_artista_proxy['std'] / duracao_por_artista_proxy['mediana']

# Artistas confiáveis: CV <= 0.5
artistas_confiaveis = duracao_por_artista_proxy[
    duracao_por_artista_proxy['cv'] <= 0.5
][['artista_unificado', 'mediana']].rename(columns={'mediana': 'duracao_artista_ms'})

print(f"Artistas com proxy confiável (CV <= 0.5): {len(artistas_confiaveis)}")

# Aplicar fontes 3a e 3b onde duracao_estimada_ms ainda é nulo
df_features_plays = df_features_plays.merge(artistas_confiaveis, on='artista_unificado', how='left')

# Fonte 3a: proxy por artista confiável
mask_3a = df_features_plays['duracao_estimada_ms'].isna() & df_features_plays['duracao_artista_ms'].notna()
df_features_plays.loc[mask_3a, 'duracao_estimada_ms'] = df_features_plays.loc[mask_3a, 'duracao_artista_ms']
df_features_plays.loc[mask_3a, 'fonte_duracao'] = 'artista'

# Fonte 3b: mediana global para o restante
mask_3b = df_features_plays['duracao_estimada_ms'].isna()
df_features_plays.loc[mask_3b, 'duracao_estimada_ms'] = mediana_global
df_features_plays.loc[mask_3b, 'fonte_duracao'] = 'global'

# Remover coluna auxiliar
df_features_plays = df_features_plays.drop(columns='duracao_artista_ms')

# Auditoria
print(f"\nDistribuição por fonte de duração (execuções):")
print(df_features_plays['fonte_duracao'].value_counts(dropna=False))
print(f"\nDistribuição por fonte de duração (URIs únicas):")
print(df_features_plays.groupby('fonte_duracao', dropna=False)['spotify_track_uri'].nunique())
print(f"\nExecuções sem duração: {df_features_plays['duracao_estimada_ms'].isna().sum()}")

Mediana global: 3.38 min
Artistas com proxy confiável (CV <= 0.5): 316

Distribuição por fonte de duração (execuções):
fonte_duracao
api    24040
Name: count, dtype: int64

Distribuição por fonte de duração (URIs únicas):
fonte_duracao
api    5374
Name: spotify_track_uri, dtype: int64

Execuções sem duração: 0


#4. Criação das features por execução

In [ ]:
# ── Features por execução ──────────────────────────────────────────────────────

# Retenção
df_features_plays['retencao'] = np.where(
    df_features_plays['duracao_estimada_ms'].notna(),
    (df_features_plays['ms_played'] / df_features_plays['duracao_estimada_ms']).clip(upper=1.0),
    np.nan
)

# Escuta completa
df_features_plays['escuta_completa'] = np.where(
    df_features_plays['retencao'].notna(),
    df_features_plays['retencao'] >= 0.65,  #Novo threshold após termos a média de 0.649
    np.nan
)

# Skip inferido por comportamento
df_features_plays['skip'] = (
    (df_features_plays['reason_end'] == 'fwdbtn') &
    (df_features_plays['retencao'] < 0.5)
)

# Início intencional
df_features_plays['inicio_intencional'] = df_features_plays['reason_start'].isin(['clickrow', 'playbtn', 'backbtn'])

# Diagnóstico
print(f"Retenção média geral: {df_features_plays['retencao'].mean():.3f}")
print(f"Escutas completas:    {df_features_plays['escuta_completa'].mean():.1%}")
print(f"Skips:                {df_features_plays['skip'].mean():.1%}")
print(f"Inícios intencionais: {df_features_plays['inicio_intencional'].mean():.1%}")
print(f"\nExecuções sem retenção (sem duração): {df_features_plays['retencao'].isna().sum()}")

Retenção média geral: 0.649
Escutas completas:    60.8%
Skips:                19.5%
Inícios intencionais: 24.3%

Execuções sem retenção (sem duração): 0


In [ ]:
# ── Agregação por música ───────────────────────────────────────────────────────

df_agg = df_features_plays.groupby(['artista_unificado', 'musica_unificada', 'spotify_track_uri']).agg(
    total_plays=('ts', 'count'),
    total_minutos=('minutos_tocados', 'sum'),
    retencao_media=('retencao', 'mean'),
    escuta_completa_rate=('escuta_completa', 'mean'),
    skip_rate=('skip', 'mean'),
    inicio_intencional_rate=('inicio_intencional', 'mean'),
    duracao_estimada_ms=('duracao_estimada_ms', 'first'),
    fonte_duracao=('fonte_duracao', 'first'),
    data_release=('data_release', 'first'),
    explicit=('explicit', 'first'),
).reset_index()

# Merge com df_ranking_final para trazer rating_lealdade_artista e demais features de artista
cols_artistas = ['artista_unificado', 'rating_lealdade_artista',
                 'longevidade_dias_artista', 'total_plays_artista']
df_ranking_final = df_agg.merge(
    df_features_plays[cols_artistas].drop_duplicates('artista_unificado'),
    on='artista_unificado',
    how='left'
)

# Diagnóstico
print(f"Músicas no ranking final: {len(df_ranking_final)}")
print(f"\nColunas: {list(df_ranking_final.columns)}")

Músicas no ranking final: 5382

Colunas: ['artista_unificado', 'musica_unificada', 'spotify_track_uri', 'total_plays', 'total_minutos', 'retencao_media', 'escuta_completa_rate', 'skip_rate', 'inicio_intencional_rate', 'duracao_estimada_ms', 'fonte_duracao', 'data_release', 'explicit', 'rating_lealdade_artista', 'longevidade_dias_artista', 'total_plays_artista']


In [ ]:
print(f"URIs únicas no df:              {df_features_plays['spotify_track_uri'].nunique()}")
print(f"Músicas no df_ranking_final:    {len(df_ranking_final)}")

URIs únicas no df:              5374
Músicas no df_ranking_final:    5382


In [ ]:
# Músicas com mais de uma URI
duplicatas = (
    df_ranking_final.groupby(['artista_unificado', 'musica_unificada'])['spotify_track_uri']
    .nunique()
    .reset_index()
    .rename(columns={'spotify_track_uri': 'qtd_uris'})
    .query('qtd_uris > 1')
    .sort_values('qtd_uris', ascending=False)
)

print(f"Músicas com mais de uma URI: {len(duplicatas)}")
display(duplicatas.head(20))

Músicas com mais de uma URI: 903


,artista_unificado,musica_unificada,qtd_uris
1396,Engenheiros e Humberto Gessinger,Somos Quem Podemos Ser,8
1258,Engenheiros e Humberto Gessinger,Eu Que Não Amo Você,6
2047,Legião Urbana,Daniel Na Cova Dos Leões,5
1180,Engenheiros e Humberto Gessinger,A Montanha,5
2915,Pink Floyd,Echoes,5
2053,Legião Urbana,Eu Sei,5
2062,Legião Urbana,Há Tempos,5
1794,Inocentes,Rotina,5
1363,Engenheiros e Humberto Gessinger,Pra Ser Sincero,5
1280,Engenheiros e Humberto Gessinger,Infinita Highway,5


In [ ]:
# URI mais escutada por música
uri_dominante = (
    df_ranking_final.groupby(['artista_unificado', 'musica_unificada'])
    .apply(lambda x: x.loc[x['total_plays'].idxmax(), 'spotify_track_uri'], include_groups=False)
    .reset_index()
    .rename(columns={0: 'spotify_track_uri_dominante'})
)

# Média ponderada segura
def media_ponderada(values, weights):
    mask = values.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

# Consolidar por musica_unificada + artista_unificado
def agregar_musica(x):
    plays = x['total_plays']
    return pd.Series({
        'total_plays':              plays.sum(),
        'total_minutos':            x['total_minutos'].sum(),
        'retencao_media':           media_ponderada(x['retencao_media'], plays),
        'escuta_completa_rate':     media_ponderada(x['escuta_completa_rate'], plays),
        'skip_rate':                media_ponderada(x['skip_rate'], plays),
        'inicio_intencional_rate':  media_ponderada(x['inicio_intencional_rate'], plays),
        'duracao_estimada_ms':      media_ponderada(x['duracao_estimada_ms'], plays),
        'fonte_duracao':            x['fonte_duracao'].value_counts().index[0] if x['fonte_duracao'].notna().any() else np.nan,
        'data_release':             x['data_release'].min(),
        'explicit':                 x['explicit'].max(),
        'rating_lealdade_artista':  x['rating_lealdade_artista'].iloc[0],
        'longevidade_dias_artista': x['longevidade_dias_artista'].iloc[0],
        'total_plays_artista':      x['total_plays_artista'].iloc[0],
    })

df_ranking_final = (
    df_ranking_final.groupby(['artista_unificado', 'musica_unificada'])
    .apply(agregar_musica, include_groups=False)
    .reset_index()
)

# Trazer URI dominante
df_ranking_final = df_ranking_final.merge(uri_dominante, on=['artista_unificado', 'musica_unificada'], how='left')

print(f"Músicas após consolidação: {len(df_ranking_final)}")

Músicas após consolidação: 4212


#5 - Exportações

In [ ]:
# ── Salvar parquets do NB3 ─────────────────────────────────────────────────────
df_features_plays.to_parquet(f'{caminho_pasta}/df_musicas_enriquecido.parquet', index=False)
df_ranking_final.to_parquet(f'{caminho_pasta}/df_ranking_final.parquet', index=False)

print(f'df_musicas_enriquecido: {df_features_plays.shape}')
print(f'df_ranking_final:       {df_ranking_final.shape}')

df_musicas_enriquecido: (24040, 39)
df_ranking_final:       (4212, 16)


In [ ]:
df_features_plays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24040 entries, 0 to 24039
Data columns (total 39 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   ts                                 24040 non-null  object             
 1   ms_played                          24040 non-null  int64              
 2   master_metadata_track_name         24040 non-null  object             
 3   master_metadata_album_artist_name  24040 non-null  object             
 4   master_metadata_album_album_name   24040 non-null  object             
 5   spotify_track_uri                  24040 non-null  object             
 6   reason_start                       24040 non-null  object             
 7   reason_end                         24040 non-null  object             
 8   shuffle                            24040 non-null  bool               
 9   skipped                            24040 non-null 

In [ ]:
df_ranking_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4212 entries, 0 to 4211
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   artista_unificado            4212 non-null   object 
 1   musica_unificada             4212 non-null   object 
 2   total_plays                  4212 non-null   int64  
 3   total_minutos                4212 non-null   float64
 4   retencao_media               4212 non-null   float64
 5   escuta_completa_rate         4212 non-null   float64
 6   skip_rate                    4212 non-null   float64
 7   inicio_intencional_rate      4212 non-null   float64
 8   duracao_estimada_ms          4212 non-null   float64
 9   fonte_duracao                4212 non-null   object 
 10  data_release                 4212 non-null   object 
 11  explicit                     4212 non-null   bool   
 12  rating_lealdade_artista      4212 non-null   float64
 13  longevidade_dias_a